# Time-to-react at a T-intersection: effect of knowing the other driver's intent

This notebook computes the time-to-react (TTR) of an automated vehicle with CommonRoad-Reach for three predictions of another vehicle's behaviour, and checks how the result depends on the tool's numerical settings.

The TTR is the latest moment along the vehicle's intended trajectory at which an evasive manoeuvre still exists, that is, the latest start time from which the vehicle's reachable set at the end of a 3 s horizon is non-empty (Söntges, Koschi and Althoff, "Worst-case Analysis of the Time-To-React Using Reachable Sets", IEEE Intelligent Vehicles Symposium 2018). As in that paper's Algorithm 1, the value reported is the first start time, in steps of 0.1 s, at which that reachable set is empty. CommonRoad-Reach can discard small regions that are only partly blocked, so its values are close estimates rather than certified upper bounds; the grid over-approximation in `commonroad_ttr.py` gives the upper bound.

**Scenario.** CommonRoad scenario `DEU_Ffb-2_2_S-1` (hand-crafted, format 2018b, with set-based predictions). The ego vehicle drives straight through a T-intersection at 14 m/s while a vehicle on the side road (obstacle 2) may turn either way. Its prediction has two branches:

| Case | Prediction of obstacle 2 |
|---|---|
| `full` | both branches: intent unknown |
| `keep20` | branch 0 only: turns in the ego vehicle's direction of travel |
| `keep21` | branch 1 only: turns across the ego vehicle's path |

Section 6 also computes the `keep20` case when the intent becomes known only after the vehicle has started, to find how late the information can arrive and still extend the TTR.

**Model.** Point mass in Cartesian coordinates with |a_x|, |a_y| ≤ 10 m/s² and |v_x|, |v_y| ≤ 14 m/s, the ego vehicle as a disc of radius 0.9 m, road boundaries enforced, time step 0.1 s, following Söntges, Koschi and Althoff (2018, Table I). At step k, the ego vehicle is checked against the occupancy predicted for the interval [k−1, k].

**Numerical settings.** CommonRoad-Reach removes any colliding rectangle whose diagonal is below `radius_terminal_split` (default 0.7 m). The default trades precision for speed, and smaller values resolve more of the free space next to obstacles, so the notebook repeats each case with smaller values and with a finer drivable-area grid (`size_grid`).

**Requirements.** A CPU runtime is sufficient. CommonRoad-Reach 2025.2.1 is distributed for Python 3.10 and 3.11, so it is installed in a separate Python 3.11 environment. `commonroad-io` is pinned to 2024.3, which this release of CommonRoad-Reach requires.

## 1. Installation

In [1]:
!pip -q install uv
!uv python install 3.11 -q
!rm -rf /content/cr && uv venv /content/cr -p 3.11 -q
!uv pip install -q --python /content/cr/bin/python commonroad-reach==2025.2.1 commonroad-io==2024.3 shapely==2.2.0 numpy==2.4.6
!/content/cr/bin/python -c "import commonroad_reach.pycrreach, commonroad_dc.pycrcc; print('CommonRoad-Reach installed')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 41.6 MB/s eta 0:00:00
CommonRoad-Reach installed


## 2. Input files

The notebook needs the script `commonroad_reach_crosscheck.py` and the scenario `DEU_Ffb-2_2_S-1.xml`. Both are available in the project's public repository, and the scenario is also available from the [CommonRoad scenario repository](https://gitlab.lrz.de/tum-cps/commonroad-scenarios).

The cell below downloads both files. If a download fails, download the file manually and upload it when prompted.

In [2]:
import re
from pathlib import Path
from urllib.request import urlretrieve
from google.colab import files

SCRIPT = "commonroad_reach_crosscheck.py"
SCENARIO = "DEU_Ffb-2_2_S-1.xml"
SOURCES = {
    SCRIPT: "https://raw.githubusercontent.com/eism-science/recoverability/v1.0/commonroad/code/"
            "commonroad_reach_crosscheck.py",
    SCENARIO: "https://gitlab.lrz.de/tum-cps/commonroad-scenarios/-/raw/master/scenarios/hand-crafted/"
              "DEU_Ffb-2_2_S-1.xml",
}

for name, url in SOURCES.items():
    if Path(name).exists():
        continue
    try:
        urlretrieve(url, name)
        print(f"Downloaded {name}")
    except Exception as error:
        print(f"Could not download {name} ({error}); please upload it.")

missing = [name for name in SOURCES if not Path(name).exists()]
if missing:
    for name, data in files.upload().items():
        Path(re.sub(r" \(\d+\)(?=\.[^.]+$)", "", name)).write_bytes(data)  # Colab renames repeated uploads to "name (1).ext"

missing = [name for name in SOURCES if not Path(name).exists()]
assert not missing, f"Missing input files: {missing}"
print("Input files ready.")

Could not download commonroad_reach_crosscheck.py (HTTP Error 404: Not Found); please upload it.
Downloaded DEU_Ffb-2_2_S-1.xml


Saving commonroad_reach_crosscheck.py to commonroad_reach_crosscheck.py
Input files ready.


## 3. Scenario check

Builds each case without computing reachable sets and reports the time to collision along the intended trajectory: 10 steps (1.0 s) for `full` and `keep21`, and 12 steps (1.2 s) for `keep20`.

In [3]:
import os
import subprocess

PYTHON = "/content/cr/bin/python"
ENV = {**os.environ, "MPLBACKEND": "Agg"}
CASES = {"full": [], "keep20": ["--keep", "2:0"], "keep21": ["--keep", "2:1"]}

for case, options in CASES.items():
    out = subprocess.run([PYTHON, SCRIPT, SCENARIO, "--build-only", *options], env=ENV,
                         capture_output=True, text=True, check=True).stdout
    print(f"{case:7s}", next(line for line in out.splitlines() if line.startswith("scenario built")))

full    scenario built: 12 lanelets, 3 obstacles; TTC steps: 10
keep20  scenario built: 12 lanelets, 3 obstacles; TTC steps: 12
keep21  scenario built: 12 lanelets, 3 obstacles; TTC steps: 10


## 4. Time-to-react

Each case is computed with the default settings (grid 0.2 m, split radius 0.7 m), a finer grid, and two smaller split radii. Each run writes a log named after its case and settings in tenths of a metre, for example `reach_keep20_grid02_split01.log`.

In [4]:
LOGS = Path("logs")
LOGS.mkdir(exist_ok=True)
SETTINGS = [(0.2, 0.7), (0.1, 0.7), (0.2, 0.2), (0.2, 0.1)]  # (size_grid, radius_terminal_split) in metres

def tenths(x):
    return f"{round(x * 10):02d}"

results = []
for case, options in CASES.items():
    for grid, split in SETTINGS:
        log = LOGS / f"reach_{case}_grid{tenths(grid)}_split{tenths(split)}.log"
        proc = subprocess.run([PYTHON, SCRIPT, SCENARIO, *options, "--grid", str(grid), "--split", str(split)],
                              env=ENV, capture_output=True, text=True)
        log.write_text(proc.stdout + proc.stderr)
        if proc.returncode != 0:
            raise RuntimeError(f"Run failed; see {log}")
        ttr = re.search(r"TTR_max = (.+)", proc.stdout).group(1)
        results.append({"case": case, "size_grid (m)": grid, "split radius (m)": split, "TTR": ttr})
        print(f"{log.name}: TTR = {ttr}")

reach_full_grid02_split07.log: TTR = 0.4 s
reach_full_grid01_split07.log: TTR = 0.4 s
reach_full_grid02_split02.log: TTR = 0.4 s
reach_full_grid02_split01.log: TTR = 0.4 s
reach_keep20_grid02_split07.log: TTR = 0.5 s
reach_keep20_grid01_split07.log: TTR = 0.5 s
reach_keep20_grid02_split02.log: TTR = 0.8 s
reach_keep20_grid02_split01.log: TTR = 0.8 s
reach_keep21_grid02_split07.log: TTR = 0.4 s
reach_keep21_grid01_split07.log: TTR = 0.4 s
reach_keep21_grid02_split02.log: TTR = 0.4 s
reach_keep21_grid02_split01.log: TTR = 0.4 s


## 5. Results

In [5]:
import pandas as pd

pd.DataFrame(results).pivot_table(index=["size_grid (m)", "split radius (m)"], columns="case",
                                  values="TTR", aggfunc="first")

case                             full keep20 keep21
size_grid (m) split radius (m)                     
0.1           0.7               0.4 s  0.5 s  0.4 s
0.2           0.1               0.4 s  0.8 s  0.4 s
              0.2               0.4 s  0.8 s  0.4 s
              0.7               0.4 s  0.5 s  0.4 s

With split radii of 0.2 m and 0.1 m, the TTR is 0.4 s when the intent is unknown (`full`) and 0.8 s when the side-road vehicle is known to turn in the ego vehicle's direction (`keep20`); knowing that it turns across (`keep21`) leaves the TTR at 0.4 s. At the default split radius of 0.7 m, `keep20` reads 0.5 s, and a radius of 0.2 m or less is needed to resolve the free space next to the side-road vehicle. Söntges, Koschi and Althoff (2018) report an upper bound of 0.5 s and an achieved value of 0.3 s for the corresponding scenario with the full prediction. An independent grid-based over-approximation (`commonroad_ttr.py`, 0.125 m grid) gives 0.5 s, 0.9 s and 0.5 s for the three cases.

## 6. Intent known later

In the `keep20` case above, the turn is known from the start. Here it becomes known at a later time: the full prediction applies to start times before that moment and the narrowed prediction from it onwards (option `--from-step`, in steps of 0.1 s). The runs use a grid of 0.2 m and a split radius of 0.2 m, which give the same values as 0.1 m above. If the information is still useful when it arrives, the TTR is that of `keep20`; if it arrives too late, the TTR is that of `full`. Because the prediction changes partway through, a later start time can again have an evasive manoeuvre once the turn is known, so every start time up to a collision is computed and logged. The TTR reported is the first start time with no evasive manoeuvre under the information available then: the vehicle must keep an evasive manoeuvre available at every moment, so once none exists it must already have started one.

In [6]:
import pandas as pd

LATER_STEPS = [2, 3, 4, 5, 6]  # time the turn becomes known, in steps of 0.1 s
later = []
for step in LATER_STEPS:
    log = LOGS / f"reach_keep20_from{step:02d}_grid02_split02.log"
    proc = subprocess.run([PYTHON, SCRIPT, SCENARIO, "--keep", "2:0", "--from-step", str(step),
                           "--grid", "0.2", "--split", "0.2"], env=ENV, capture_output=True, text=True)
    log.write_text(proc.stdout + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Run failed; see {log}")
    ttr = re.search(r"TTR_max = (.+)", proc.stdout).group(1)
    later.append({"turn known at (s)": round(step * 0.1, 1), "TTR": ttr})
    print(f"{log.name}: TTR = {ttr}")

pd.DataFrame(later)

reach_keep20_from02_grid02_split02.log: TTR = 0.8 s
reach_keep20_from03_grid02_split02.log: TTR = 0.8 s
reach_keep20_from04_grid02_split02.log: TTR = 0.8 s
reach_keep20_from05_grid02_split02.log: TTR = 0.4 s
reach_keep20_from06_grid02_split02.log: TTR = 0.4 s


,turn known at (s),TTR
0,0.2,0.8 s
1,0.3,0.8 s
2,0.4,0.8 s
3,0.5,0.4 s
4,0.6,0.4 s


## 6a. Further checks of the numerical settings


Two checks of the values in Section 4, for `full` and `keep20`. First, a finer grid (0.1 m) with a smaller split radius (0.05 m), to see whether the `keep20` value is stable beyond the settings above. Second, the settings of Table 2 (grid 0.2 m, split radius 0.2 m) with the initial position and velocity uncertainty set to 10⁻⁶ instead of Reach's defaults, so that each computation starts from a practically exact state; the defaults are printed in the logs. Logs are named with settings in hundredths of a metre, for example `reach_keep20_grid010_split005.log`.

In [7]:
import pandas as pd
LOGS = Path("logs")
LOGS.mkdir(exist_ok=True)

def hundredths(x):
    return f"{round(x * 100):03d}"

CHECKS = [(0.1, 0.05, None), (0.2, 0.2, 1e-6)]  # (size_grid, radius_terminal_split, initial uncertainty)
checks = []
for case in ["full", "keep20"]:
    for grid, split, unc in CHECKS:
        extra = [] if unc is None else ["--uncertainty", str(unc)]
        log = LOGS / (f"reach_{case}_grid{hundredths(grid)}_split{hundredths(split)}"
                      + ("" if unc is None else "_exact") + ".log")
        proc = subprocess.run([PYTHON, SCRIPT, SCENARIO, *CASES[case], "--grid", str(grid), "--split", str(split),
                               *extra], env=ENV, capture_output=True, text=True)
        log.write_text(proc.stdout + proc.stderr)
        if proc.returncode != 0:
            raise RuntimeError(f"Run failed; see {log}")
        ttr = re.search(r"TTR_max = (.+)", proc.stdout).group(1)
        unc_line = next((l for l in proc.stdout.splitlines() if l.startswith("initial uncertainty")), "")
        checks.append({"case": case, "size_grid (m)": grid, "split radius (m)": split,
                       "initial uncertainty": "Reach default" if unc is None else unc, "TTR": ttr})
        print(f"{log.name}: TTR = {ttr}   [{unc_line}]")

pd.DataFrame(checks)

reach_full_grid010_split005.log: TTR = 0.4 s   [initial uncertainty: default uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncertainty_v_lat=0.01 | used uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncertainty_v_lat=0.01]
reach_full_grid020_split020_exact.log: TTR = 0.4 s   [initial uncertainty: default uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncertainty_v_lat=0.01 | used uncertainty_p_lon=1e-06, uncertainty_p_lat=1e-06, uncertainty_v_lon=1e-06, uncertainty_v_lat=1e-06]
reach_keep20_grid010_split005.log: TTR = 0.8 s   [initial uncertainty: default uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncertainty_v_lat=0.01 | used uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncertainty_v_lat=0.01]
reach_keep20_grid020_split020_exact.log: TTR = 0.8 s   [initial uncertainty: default uncertainty_p_lon=0.01, uncertainty_p_lat=0.01, uncertainty_v_lon=0.01, uncert

,case,size_grid (m),split radius (m),initial uncertainty,TTR
0,full,0.1,0.05,Reach default,0.4 s
1,full,0.2,0.20,0.000001,0.4 s
2,keep20,0.1,0.05,Reach default,0.8 s
3,keep20,0.2,0.20,0.000001,0.8 s


## 7. Download the logs

In [8]:
!uv pip list --python /content/cr/bin/python > logs/versions.txt
!zip -qrj reach_logs.zip logs
files.download("reach_logs.zip")

Using Python 3.11.17 environment at: cr


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>